In [1]:
library(data.table)
library(dplyr)
library(openxlsx)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:data.table’:

    between, first, last


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




# Output supplementary tables folder

In [2]:
# converts GSEA results from .rds to .xlsx
# generates summary tables and outputs as .xlsx
dirs <- list.dirs("./results", recursive = FALSE)
out_dir <- file.path(".", "supplementary_tables")
dir.create(out_dir, showWarnings = FALSE)

counter <- 2

for (dir_path in dirs) {
    rds_files <- list.files(dir_path, pattern = "\\.rds$", full.names = TRUE)

    for (rds_file in rds_files) {
        data <- readRDS(rds_file)

        # save GSEA results as .xlsx
        wb <- createWorkbook()
        addWorksheet(wb, "data")
        writeData(wb, "data", data)
        base_name <- tools::file_path_sans_ext(basename(rds_file))
        out_file  <- file.path(out_dir, paste0("S", counter, "_", base_name, ".xlsx"))
        saveWorkbook(wb, out_file, overwrite = TRUE)
        counter <- counter + 1

        # generate summary table
        summary_df <- data %>%
            dplyr::group_by(pathway) %>%
            dplyr::summarise(
                mean_padj = mean(padj[padj < 0.05], na.rm = TRUE),
                N         = sum(padj < 0.05, na.rm = TRUE),
                N_pos     = sum(padj < 0.05 & ES > 0, na.rm = TRUE),
                N_neg     = sum(padj < 0.05 & ES < 0, na.rm = TRUE),
                size      = mean(size),
                .groups   = "drop"
            ) %>%
            arrange(mean_padj)

        # save summary as .xlsx
        wb <- createWorkbook()
        addWorksheet(wb, "summary")
        writeData(wb, "summary", summary_df)
        base_name <- tools::file_path_sans_ext(basename(rds_file))
        out_file  <- file.path(out_dir, paste0("S", counter, "_", base_name, "_summary",".xlsx"))
        saveWorkbook(wb, out_file, overwrite = TRUE)
        
        counter <- counter + 1
    }
}

# View summary tables in notebook

In [3]:
# you can uncomment one of these .rds files and then view the summary table by running the following cell

## Cancer groups files
# data <- readRDS("./results/per_sample_gsea/normalized_Basenji_cancer_per_sample_GSEA.rds") 
# data <- readRDS("./results/per_sample_gsea/normalized_Enformer_cancer_per_sample_GSEA.rds") 

## Tissue groups files
# data <- readRDS("./results/per_sample_gsea/normalized_Basenji_tissue_per_sample_GSEA.rds") 
# data <- readRDS("./results/per_sample_gsea/normalized_Enformer_tissue_per_sample_GSEA.rds")

## Tissue & cancer groups files
# data <- readRDS("./results/per_sample_gsea/normalized_Basenji_tissue_and_cancer_per_sample_GSEA.rds") 
# data <- readRDS("./results/per_sample_gsea/normalized_Enformer_tissue_and_cancer_per_sample_GSEA.rds") 

## Cancer differential files
# data <- readRDS("./results/differential_gsea/normalized_Basenji_cancer_differential_GSEA.rds") 
data <- readRDS("./results/differential_gsea/normalized_Enformer_cancer_differential_GSEA.rds") 


In [4]:
data %>%
    dplyr::group_by(pathway) %>%
    dplyr::summarise(
    mean_padj = mean(padj[padj<0.05]),
    N     = sum(padj < 0.05, na.rm = TRUE),
    N_pos = sum(padj < 0.05 & ES > 0, na.rm = TRUE),
    N_neg = sum(padj < 0.05 & ES < 0, na.rm = TRUE),
    size = mean(size),
    ) %>%arrange(mean_padj)

pathway,mean_padj,N,N_pos,N_neg,size
<chr>,<dbl>,<int>,<int>,<int>,<dbl>
GOMF_G_PROTEIN_COUPLED_RECEPTOR_ACTIVITY,0.0002923361,1,0,1,92
GOBP_CIRCULATORY_SYSTEM_PROCESS,0.0004941353,1,0,1,54
GOBP_DETECTION_OF_CHEMICAL_STIMULUS,0.0004941353,1,0,1,61
GOBP_SENSORY_PERCEPTION_OF_CHEMICAL_STIMULUS,0.0004941353,1,0,1,61
GOMF_TRANSCRIPTION_REGULATOR_ACTIVITY,0.0004941353,1,1,0,320
GOBP_DETECTION_OF_STIMULUS_INVOLVED_IN_SENSORY_PERCEPTION,0.0010640952,1,0,1,60
GOBP_REGULATION_OF_IMMUNE_RESPONSE,0.0010640952,1,0,1,112
GOMF_OLFACTORY_RECEPTOR_ACTIVITY,0.0010938602,1,0,1,43
GOBP_SENSORY_PERCEPTION_OF_SMELL,0.0013574411,1,0,1,46
